# Ontology Design (concepts + relationships)

**Concepts (Classes):**
- Student
- Course

**Relationships (Properties):**
- requires (Course → Course) – course prerequisites
- completed (Student → Course) – courses a student has completed
- eligibleFor (Student → Course) – inferred relationship showing course eligibility


In [1]:


class OntologyUniversity:
    def __init__(self):

        self.students=set()
        self.courses=set()
        self.prerequisites = {}

        self.completed = set()
        self.eligible = set()

    def add_students(self, students):
        self.students.add(students)

    def add_courses(self, courses):
        self.courses.add(courses)

    def add_prerequisite(self, course, prereq):
        if course not in self.prerequisites:
            self.prerequisites[course] = set()
        self.prerequisites[course].add(prereq)

    def complete_course(self, student, course):
        self.completed.add((student, course))

    def infer_eligibility(self, student, course):
        if student not in self.students:
            raise ValueError("Unknown student")
        if course not in self.courses:
            raise ValueError("Unknown course")

        required = self.prerequisites.get(course, set())
        completed_courses = {c for (s, c) in self.completed if s == student}

        missing = required - completed_courses

        if not missing:
            self.eligible.add((student, course))
            return True, set()

        return False, missing
    
    def recommend_courses(self, students):
        if students not in self.students:
            raise ValueError("Unknown student")

        completed_courses = {c for (s, c) in self.completed if s == students}
        recommendations = []

        for course in self.courses:
            if course not in completed_courses:
                can_take, _ = self.infer_eligibility(students, course)
                if can_take:
                    recommendations.append(course)

        return sorted(recommendations)


## Dataset

Students: Erick, Tarah, Stacy  
Courses: Calculus 1, Calculus 2, Discrete Math, Probability and Statistics, Linear Algebra, Machine Learning  

Prerequisites:  
- Calculus 2 requires Calculus 1  
- Discrete Math requires Calculus 2  
- Probability and Statistics requires Discrete Math  
- Machine Learning requires Discrete Math and Probability and Statistics  
- Calculus 1 requires Linear Algebra


## Inference Demo (eligibility checks + explanations)


In [2]:



# ---- Usage ----
ou = OntologyUniversity()



# Add Students
ou.add_students("Erick")
ou.add_students("Tarah")
ou.add_students("Stacy")


#Add Courses
ou.add_courses("Calculus 1")
ou.add_courses("Calculus 2")
ou.add_courses("Discrete Math")
ou.add_courses("Probability and Statistics")
ou.add_courses("Linear Algebra")
ou.add_courses("Machine Learning")

# Add Prerequisites
ou.add_prerequisite("Calculus 1", "Linear Algebra")
ou.add_prerequisite("Calculus 2", "Calculus 1")
ou.add_prerequisite("Discrete Math", "Calculus 2")
ou.add_prerequisite("Probability and Statistics", "Discrete Math")
ou.add_prerequisite("Machine Learning", "Discrete Math")
ou.add_prerequisite("Machine Learning", "Probability and Statistics")
ou.add_prerequisite("Artificial Intelligence", "Statistics")
ou.add_prerequisite("Calculus  2", "Linear Algebra")

# Complete Courses
ou.complete_course("Erick", "Linear Algebra")
ou.complete_course("Erick", "Calculus 1")

ou.complete_course("Tarah", "Linear Algebra")
ou.complete_course("Tarah", "Calculus 1")
ou.complete_course("Tarah", "Calculus 2")

ou.complete_course("Stacy", "Linear Algebra")
ou.complete_course("Stacy", "Calculus 1")
ou.complete_course("Stacy", "Calculus 2")
ou.complete_course("Stacy", "Discrete Math")

# Infer Eligibility
print("Tarah eligibility for Discrete Math:", ou.infer_eligibility("Tarah", "Discrete Math"))
print("Erick eligibility for Discrete Math:", ou.infer_eligibility("Erick", "Discrete Math"))
print("Stacy eligibility for Probability and Statistics:", ou.infer_eligibility("Stacy", "Probability and Statistics"))
print("Stacy eligibility for Machine Learning:", ou.infer_eligibility("Stacy", "Machine Learning"))


Tarah eligibility for Discrete Math: (True, set())
Erick eligibility for Discrete Math: (False, {'Calculus 2'})
Stacy eligibility for Probability and Statistics: (True, set())
Stacy eligibility for Machine Learning: (False, {'Probability and Statistics'})


## Recommendations


In [3]:

# Recommend Courses
print("Tarah course recommendations:", ou.recommend_courses("Tarah"))
print("Erick course recommendations:", ou.recommend_courses("Erick"))
print("Stacy course recommendations:", ou.recommend_courses("Stacy"))

Tarah course recommendations: ['Discrete Math']
Erick course recommendations: ['Calculus 2']
Stacy course recommendations: ['Probability and Statistics']


## Unit Tests


In [4]:
import unittest

class TestOntologyUniversity(unittest.TestCase):
    
    def setUp(self):
        # Simple setup: small dataset
        self.ou = OntologyUniversity()
        # Students
        self.ou.add_students("Erick")
        self.ou.add_students("Tarah")
        self.ou.add_students("Stacy")
        # Courses
        self.ou.add_courses("Calculus 1")
        self.ou.add_courses("Calculus 2")
        self.ou.add_courses("Discrete Math")
        self.ou.add_courses("Probability and Statistics")
        # Prerequisites
        self.ou.add_prerequisite("Calculus 2", "Calculus 1")
        self.ou.add_prerequisite("Discrete Math", "Calculus 2")
        self.ou.add_prerequisite("Probability and Statistics", "Discrete Math")
        # Completed courses
        self.ou.complete_course("Erick", "Calculus 1")
        self.ou.complete_course("Tarah", "Calculus 1")
        self.ou.complete_course("Tarah", "Calculus 2")
        self.ou.complete_course("Stacy", "Calculus 1")
        self.ou.complete_course("Stacy", "Calculus 2")
        self.ou.complete_course("Stacy", "Discrete Math")

    # --- Eligibility Tests ---
    def test_eligibility_true(self):
        # Tarah should be eligible for Discrete Math
        eligible, missing = self.ou.infer_eligibility("Tarah", "Discrete Math")
        self.assertTrue(eligible)
        self.assertEqual(missing, set())

    def test_eligibility_false(self):
        # Erick missing Calculus 2 for Discrete Math
        eligible, missing = self.ou.infer_eligibility("Erick", "Discrete Math")
        self.assertFalse(eligible)
        self.assertEqual(missing, {"Calculus 2"})

    # --- Recommendation Tests ---
    def test_recommendations(self):
        # Stacy should be recommended Probability and Statistics
        recs = self.ou.recommend_courses("Stacy")
        self.assertEqual(recs, ["Probability and Statistics"])
    
    def test_no_recommendations_yet(self):
        # Erick should only have Calculus 2 as recommendation
        recs = self.ou.recommend_courses("Erick")
        self.assertEqual(recs, ["Calculus 2"])

# Run all tests
unittest.main(argv=[''], exit=False)


....
----------------------------------------------------------------------
Ran 4 tests in 0.005s

OK
